In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
import gc


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load the dataset
file_path = '/content/drive/My Drive/Dataset/tweets.csv'
df = pd.read_csv(file_path, encoding='latin-1', header=None)

# Define column headers
df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']

# Convert target labels from 0 and 4 to 0 and 1
df['target'] = df['target'].replace(4, 1)

# Text Cleaning Function
def clean_text(text):
    import re
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Remove URLs
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = text.lower()  # Convert to lowercase
    return text

# Apply text cleaning
df['cleaned_text'] = df['text'].apply(clean_text)

# Class balance check
print("\nClass Balance (Sentiment Labels):")
print(df['target'].value_counts())



Class Balance (Sentiment Labels):
target
0    800000
1    800000
Name: count, dtype: int64


In [ ]:
# Download NLTK resources
nltk.download('stopwords')

# Text Cleaning Function
def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Remove URLs
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = text.lower()  # Convert to lowercase
    return text

# Apply text cleaning
df['cleaned_text'] = df['text'].apply(clean_text)

# Remove stopwords
stop_words = set(stopwords.words('english'))
df['cleaned_text'] = df['cleaned_text'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop_words])
)

# Stemming
stemmer = SnowballStemmer("english")
df['cleaned_text'] = df['cleaned_text'].apply(
    lambda x: ' '.join([stemmer.stem(word) for word in x.split()])
)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Tokenization
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['cleaned_text'])
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(df['cleaned_text'])
padded_sequences = pad_sequences(sequences, maxlen=100)


In [ ]:
# Load pre-trained GloVe embeddings
glove_file_path = '/content/drive/My Drive/Dataset/glove.6B.100d.txt'
embedding_index = {}
embedding_dim = 100  # GloVe embedding dimensions

with open(glove_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        values = line.split()
        word = values[0]
        coefficients = np.array(values[1:], dtype='float32')
        embedding_index[word] = coefficients

# Prepare embedding matrix
vocab_size = min(5000, len(word_index)) + 1
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in word_index.items():
    if i < vocab_size:
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

print("Embedding matrix shape:", embedding_matrix.shape)


Embedding matrix shape: (5001, 100)


In [ ]:
# Define target labels
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, y, test_size=0.2, random_state=42
)

# Further split training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)


In [ ]:
# Install Hugging Face Transformers library
!pip install transformers datasets

# Import libraries
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BartTokenizer, BartForSequenceClassification
from sklearn.metrics import accuracy_score
import pandas as pd
import gc


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        # Tokenize the text
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors="pt"
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset
texts = df['cleaned_text'].tolist()
labels = df['target'].tolist()

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Load BART tokenizer
bart_model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(bart_model_name)
train_dataset = SentimentDataset(X_train, y_train, tokenizer, max_len=128)


In [ ]:
# Create datasets
train_dataset = SentimentDataset(X_train, y_train, tokenizer, max_len=100)
val_dataset = SentimentDataset(X_val, y_val, tokenizer, max_len=100)
test_dataset = SentimentDataset(X_test, y_test, tokenizer, max_len=100)

# DataLoader
batch_size = 128  # or 128 if memory allows
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


In [ ]:
# Load pre-trained BART model
model = BartForSequenceClassification.from_pretrained(bart_model_name, num_labels=2)

# Define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Some weights of BartForSequenceClassification were not initialized from the model checkpoint at facebook/bart-base and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BartForSequenceClassification(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_la

In [ ]:
scaler = torch.amp.GradScaler()  # Updated GradScaler initialization

num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    total_loss, total_acc = 0, 0
    for batch in train_loader:
        optimizer.zero_grad()

        # Move data to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # Forward pass with updated autocast
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):  # Specify device and precision
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        # Backward pass with updated GradScaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Calculate accuracy
        preds = torch.argmax(outputs.logits, dim=1)
        acc = accuracy_score(labels.cpu(), preds.cpu())
        total_loss += loss.item()
        total_acc += acc

    print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Accuracy = {total_acc/len(train_loader):.4f}")


Epoch 1: Loss = 0.4641, Accuracy = 0.7769
Epoch 2: Loss = 0.4258, Accuracy = 0.8010
Epoch 3: Loss = 0.4052, Accuracy = 0.8132


In [ ]:
# Save the model and tokenizer
model.save_pretrained('/content/drive/My Drive/BART_Sentiment_Model')
tokenizer.save_pretrained('/content/drive/My Drive/BART_Sentiment_Model')
print("Model and tokenizer saved successfully!")

/usr/local/lib/python3.10/dist-packages/transformers/configuration_utils.py:388: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}
  warnings.warn(


Model and tokenizer saved successfully!


In [ ]:
# Evaluation
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
print("Test Accuracy:", test_acc)


Test Accuracy: 0.806734375
